# Demo 02: feature accentuation along an encoding axis

Feature accentuation (Hamblin et al.) perturbs a natural seed image so that the encoding model's predicted response
reaches a chosen **target level**, while a Fourier-space parameterization with a $1/f^{\alpha}$ gradient
preconditioner, random crops and noise keep the result natural-looking. The production code is
`scripts/synthesis/feature_viz.py` + `run.py` (it needs the `horama` package, which is not part of the reference
environment); `_demo_utils.feature_accentuation` is a line-by-line port in plain torch.

**Scale.** The paper used the `fa_hyperparameters` of `scripts/synthesis/example_config.yaml`:
noise 0.1, decay 1.5, 6000 NAdam steps at learning rate 12, a 1024-px canvas evaluated at 224 px, 8 crops per
step of 90-95 % of the canvas, 11 target levels covering the natural response range extended by
`extend_range = 0.25`. With `FULL_SCALE = False` the canvas is 256 px, at most 150 steps per level (with the
production early-stop when the target is reached), 4 crops per step, learning rate 30 (the canvas is 16x
smaller and the step budget 40x smaller, so the step size is raised), and 5 target levels.

In [1]:
import sys
sys.path[:0] = ['..', 'notebooks']        # notebooks/nbsetup.py holds the setup shared by every notebook
from nbsetup import setup

OUT_DIR = setup(needs='source')          # single-thread BLAS, Agg backend, repo on the path, data check
import os, sys, time
from pathlib import Path
from pnc import paths
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from pnc.utils import apply_figure_style
apply_figure_style(); plt.rcParams['figure.dpi'] = 90
torch.set_num_threads(max(1, os.cpu_count() // 2)); torch.manual_seed(0); np.random.seed(0)
print('source_data       =', paths.source_data())
print('preprocessed_data =', paths.preprocessed_data())
HINT = 'python data/download_data.py --tier source   (README section 2)'
from IPython.display import display

source_data       = /private/tmp/claude-501/-Users-jacobprince-KonkLab-Dropbox-Jacob-Prince-Research-Prince-Accentuate-VVS-Manuscript/bca6882d-ec2d-4dbd-beb1-0d67d8b1fcc7/scratchpad/source_data
preprocessed_data = /private/tmp/claude-501/-Users-jacobprince-KonkLab-Dropbox-Jacob-Prince-Research-Prince-Accentuate-VVS-Manuscript/bca6882d-ec2d-4dbd-beb1-0d67d8b1fcc7/scratchpad/preproc_build


In [2]:
FULL_SCALE = False
MONKEY, UNIT, MODEL = 'red', 9, 'resnet50'
SEED_IDX = 3                                   # shared0241_nsd20065.png (the cat), seed image 3 of the sweeps
PAPER = dict(noise=0.1, decay=1.5, total_steps=6000, learning_rate=12.0, image_size=1024, model_input_size=224,
             crops_per_iteration=8, box_size=(0.90, 0.95), values_range=(0.0, 1.0))
NUM_LEVELS_PAPER, EXTEND_RANGE = 11, 0.25
if FULL_SCALE:
    FA, NUM_LEVELS = dict(PAPER), NUM_LEVELS_PAPER
else:
    FA = dict(PAPER, total_steps=150, learning_rate=30.0, image_size=256, crops_per_iteration=4)
    NUM_LEVELS = 5
print(FA, '\nlevels:', NUM_LEVELS)

{'noise': 0.1, 'decay': 1.5, 'total_steps': 150, 'learning_rate': 30.0, 'image_size': 256, 'model_input_size': 224, 'crops_per_iteration': 4, 'box_size': (0.9, 0.95), 'values_range': (0.0, 1.0)} 
levels: 5


## 1. The fitted readout (post-hoc prediction pickle)

`source_data/image_pca_projections/<session>/posthoc_model_predict_PCA_popul_unit/posthoc_prediction_NSDencimg_PCA_pop_unit_<session>_unit<u>_<model>.pkl`
is written by `scripts/encoding/posthoc_prediction_natimg_unit_population_PCA.py` after the fit. It carries the
accentuation `config` (the yaml that drove the synthesis: layer name, hyperparameters, seed images), the
single-site readout (`readout_vec` in the 750-d PCA space, `readout_bias`), and the calibration images projected
into that PCA space (`PCA_resp`, 969 x 750) with the measured population and target-site responses. The PCA basis
itself is the exported JIT transform `encoding_model_outputs/Encoding_model_outputs/<session>/..._Xtfmer_<layer>_pca750_RidgeCV_JITscript.pt`
(`PCA_torch` from demo 01, scripted).

In [3]:
from pnc.utils import load_encoding_pickle, MONKEY_DIRS
import _demo_utils as U

pkl = load_encoding_pickle(MONKEY, UNIT, MODEL)
for k, v in pkl.items():
    if hasattr(v, 'shape'):
        print(f'  {k:18s} {type(v).__name__} {tuple(v.shape)}')
    elif isinstance(v, dict):
        print(f'  {k:18s} dict with keys {list(v)[:8]} ...')
    else:
        print(f'  {k:18s} {type(v).__name__}')
cfg = pkl['config']
print('\nlayer:', cfg['layer_name'], '| fit:', cfg['fit_method_name'], '| levels:', cfg['num_levels'], '| extend_range:', cfg['extend_range'])
print('fa_hyperparameters:', cfg['fa_hyperparameters'])
print('seed images:', [os.path.basename(p) for p in cfg['seed_image_paths']])

  config             dict with keys ['model_name', 'fit_method_name', 'unit_ids', 'subject_id', 'layer_name', 'outputdir', 'readout_path', 'xtransform_path'] ...
  df                 DataFrame (969, 8)
  PCA_resp           Tensor (969, 750)
  population_resp    Tensor (969, 64)
  target_unit_resp   Tensor (969, 1)
  readout_vec        Tensor (750,)
  readout_bias       Tensor ()

layer: .layer4.Bottleneck0 | fit: RidgeCV | levels: 11 | extend_range: 0.25
fa_hyperparameters: {'noise': 0.1, 'decay': 1.5, 'total_steps': 6000, 'learning_rate': 12.0, 'image_size': 1024, 'model_input_size': 224, 'values_range': [0.0, 1.0], 'crops_per_iteration': 8, 'box_size': [0.9, 0.95], 'penalty': 0.0}
seed images: ['shared0575_nsd43157.png', 'shared0850_nsd61798.png', 'shared0968_nsd70194.png', 'shared0241_nsd20065.png', 'shared0160_nsd13231.png', 'shared0070_nsd07008.png', 'shared0055_nsd05879.png', 'shared0668_nsd48623.png', 'shared0488_nsd36979.png', 'shared0940_nsd68312.png']


## 2. Backbone -> layer -> PCA -> readout as a differentiable torch objective

Everything is frozen except the input. `featureFetcher.record(..., ingraph=True)` keeps the hooked activation in
the autograd graph; the readout is a one-row `nn.Linear` (the same construction as `LinearLayer_from_sklearn` /
`generate_macos.build_readout_layer`). The check below reproduces the pickle's stored predictions for a few
calibration images.

In [4]:
from core.layer_hook_utils import featureFetcher

backbone, transform = U.load_backbone(MODEL, device='cpu')
layer = cfg['layer_name']
fetcher = featureFetcher(backbone, input_size=(3, 224, 224), device='cpu', print_module=False, store_device='cpu')
fetcher.record(layer, ingraph=True, store_device='cpu')      # ingraph: the activation stays in autograd so gradients reach the pixels
xtransform = torch.jit.load(str(paths.require(U.find_export(MONKEY, UNIT, MODEL, layer, 'Xtfmer'), HINT))).eval()
readout = torch.nn.Linear(750, 1)                            # the single-site readout in PCA space, as a frozen one-row linear layer
readout.weight.data = pkl['readout_vec'].float().view(1, -1); readout.bias.data = pkl['readout_bias'].float().view(1)
readout.requires_grad_(False)

def predict(x01):
    """[0,1] images (B,3,H,W) -> predicted standardized response (B,) of site UNIT."""
    x = torch.nn.functional.interpolate(x01, size=(224, 224), mode='bilinear', align_corners=False, antialias=True) \
        if x01.shape[-1] != 224 else x01
    backbone(U.normalize01(x))
    return readout(xtransform(fetcher[layer]))[:, 0]

names = pkl['df'].stimulus_name.values
stored = (pkl['PCA_resp'] @ pkl['readout_vec'] + pkl['readout_bias']).numpy()   # the pickle's predictions, from its stored PCA projections
idx = [0, 100, 400, 800]
with torch.no_grad():
    here = predict(torch.stack([U.load_image01(U.stimulus_path(names[i])) for i in idx])).numpy()
print(pd.DataFrame({'image': names[idx], 'stored prediction': stored[idx].round(3), 'this objective': here.round(3)}).to_string(index=False))

/private/tmp/claude-501/-Users-jacobprince-KonkLab-Dropbox-Jacob-Prince-Research-Prince-Accentuate-VVS-Manuscript/bca6882d-ec2d-4dbd-beb1-0d67d8b1fcc7/scratchpad/env/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/private/tmp/claude-501/-Users-jacobprince-KonkLab-Dropbox-Jacob-Prince-Research-Prince-Accentuate-VVS-Manuscript/bca6882d-ec2d-4dbd-beb1-0d67d8b1fcc7/scratchpad/env/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


                  image  stored prediction  this objective
      BigAnimate_20.png             -0.186          -0.190
             UFM_77.png             -0.246          -0.220
shared0164_nsd13654.png             -0.161          -0.138
shared0756_nsd55006.png             -0.255          -0.263


## 3. Target levels

`run.py::compute_unit_levels` spans the site's natural response range (1st-99th percentile of the calibration
responses, from the exported `meta` statistics) extended by `extend_range` x bandwidth below and by
2 x `extend_range` x bandwidth above. The paper used 11 levels; here `NUM_LEVELS` of them.

In [5]:
r = U.load_readout(MONKEY, UNIT, MODEL)
q01, q99 = r['stats']['q01_resp'], r['stats']['q99_resp']
levels = U.compute_unit_levels(q01, q99, extend_range=EXTEND_RANGE, num_levels=NUM_LEVELS)
paper_levels = U.compute_unit_levels(q01, q99, extend_range=EXTEND_RANGE, num_levels=NUM_LEVELS_PAPER)
print(f'natural range of site {UNIT} (q01, q99) = ({q01:.2f}, {q99:.2f})')
print('target levels (this run):', levels.round(2))
print('target levels (paper, 11):', paper_levels.round(2))
seed_path = U.stimulus_path(U.SEED_NAMES[SEED_IDX])
seed = U.load_image01(seed_path, FA['image_size'])       # loaded at canvas size; predict() resizes to 224 px for the model
with torch.no_grad():
    seed_score = float(predict(seed[None]))
print(f'seed image {seed_path.name}: predicted response {seed_score:+.3f}')

natural range of site 9 (q01, q99) = (-0.69, 3.55)
target levels (this run): [-1.75  0.11  1.96  3.81  5.67]
target levels (paper, 11): [-1.75 -1.01 -0.26  0.48  1.22  1.96  2.7   3.44  4.18  4.93  5.67]
seed image shared0241_nsd20065.png: predicted response -0.244


## 4. Run the sweep

For each target the optimizer starts from the seed's inverse spectrum and ascends the readout on random crops;
the full canvas is scored every step, the closest image to the target is kept, the loss is flipped (and damped)
once the target is overshot, and the run stops when within 0.01 of the target -- exactly `FeatureVisualizer._fa`.

In [6]:
results = []
t0 = time.time()
for lvl in levels:
    t1 = time.time()
    res = U.feature_accentuation(predict, seed, target_level=float(lvl), device='cpu', verbose=False, **FA)
    results.append(res)
    print(f'target {lvl:+.2f} -> achieved {res["score"]:+.3f} in {res["steps"]:4d} steps ({time.time() - t1:.0f} s)')
print(f'sweep total: {(time.time() - t0) / 60:.1f} min')

target -1.75 -> achieved -1.742 in   43 steps (25 s)


target +0.11 -> achieved +0.111 in   11 steps (6 s)


target +1.96 -> achieved +1.965 in   35 steps (20 s)


target +3.81 -> achieved +3.813 in   77 steps (46 s)


target +5.67 -> achieved +5.659 in  136 steps (81 s)
sweep total: 3.0 min


In [7]:
fig, axes = plt.subplots(2, NUM_LEVELS + 1, figsize=(2.2 * (NUM_LEVELS + 1), 4.6),
                         gridspec_kw=dict(height_ratios=[1, 0.55]))
axes[0, 0].imshow(U.to_display(seed)); axes[0, 0].set_title(f'seed\n{seed_score:+.2f}', fontsize=8)
for ax, res, lvl in zip(axes[0, 1:], results, levels):
    ax.imshow(U.to_display(res['image'])); ax.set_title(f'target {lvl:+.2f}\nachieved {res["score"]:+.2f}', fontsize=8)
for ax in axes[0]:
    ax.set_axis_off()
for ax in axes[1, 1:]:                                   # the bottom row holds one history panel; drop the unused slots
    ax.remove()
ax = axes[1, 0]
for res, lvl in zip(results, levels):
    ax.plot(res['history'], lw=1, label=f'{lvl:+.1f}'); ax.axhline(lvl, color='gray', lw=0.4, ls=':')
ax.set(xlabel='step', ylabel='full-canvas prediction'); ax.legend(fontsize=6, ncol=2, title='target', title_fontsize=6)
fig.suptitle(f'{MODEL} accentuation sweep, Monkey R site {UNIT}, seed {SEED_IDX} ({FA["image_size"]} px canvas, <= {FA["total_steps"]} steps/level)', y=0.99)
plt.tight_layout(); display(plt.gcf()); plt.close()

<Figure size 1188x414 with 7 Axes>

## 5. Achieved vs requested levels, and the production sweep

Left: the level the optimizer reached against the requested one. Right: the shipped stimuli of the same
site / model / seed from the full-scale run (`stimuli_control/`), whose file names carry `level` and `score`.

In [8]:
from pnc.utils import STIMULI_CONTROL_PATH, ACCENT_DATE_PREFIXES, parse_stimulus_name
from PIL import Image
d = Path(STIMULI_CONTROL_PATH) / f'{ACCENT_DATE_PREFIXES[MONKEY]}_{MONKEY_DIRS[MONKEY]}_{MODEL}_accentuation'
prod = sorted([dict(path=p, **parse_stimulus_name(p.name)) for p in d.glob('*.png')
               if parse_stimulus_name(p.name) and parse_stimulus_name(p.name)['unit'] == UNIT
               and parse_stimulus_name(p.name)['seed'] == SEED_IDX], key=lambda q: q['target'])

fig = plt.figure(figsize=(14, 3.4)); gs = fig.add_gridspec(1, 2, width_ratios=[1, 3.2])
ax = fig.add_subplot(gs[0])
ax.plot([lv for lv in levels], [rs['score'] for rs in results], 'o', color='C3', label=f'this notebook ({FA["image_size"]} px, <= {FA["total_steps"]} steps)')
ax.plot([q['target'] for q in prod], [q['score'] for q in prod], 's', mfc='none', color='C0', label='production (1024 px, 6000 steps)')
lo, hi = min(levels.min(), prod[0]['target']) - 0.3, max(levels.max(), prod[-1]['target']) + 0.3
# identity line, and the natural range that the outer levels deliberately exceed
ax.plot([lo, hi], [lo, hi], 'k--', lw=0.6); ax.axvspan(q01, q99, color='gray', alpha=0.12, label='natural range (q01-q99)')
ax.set(xlabel='requested level', ylabel='achieved predicted level', title='target tracking'); ax.legend(fontsize=6)
sub = fig.add_gridspec(1, len(prod), left=0.32, right=0.99, top=0.85, bottom=0.05)
for i, q in enumerate(prod):
    a = fig.add_subplot(sub[i]); a.imshow(Image.open(q['path']).convert('RGB')); a.set_axis_off()
    a.set_title(f"{q['target']:+.1f} / {q['score']:+.2f}", fontsize=6)
fig.text(0.65, 0.95, 'production sweep (target / achieved)', ha='center', fontsize=9)
display(plt.gcf()); plt.close()

<Figure size 1260x306 with 12 Axes>

## 6. MACO-style unconstrained maximization

Without a seed image the same objective can be maximized from scratch. MACO (Fel et al.) fixes the Fourier
magnitude to a natural-image template and optimizes only the phase; `scripts/synthesis/generate_macos.py` ran
`horama.maco` for 4096 steps on a 2048-px canvas (12 crops of 20-25 %, noise 0.08, lr 0.1). Here the template is
the mean magnitude spectrum of 64 calibration images at the working canvas size and the run is 100 steps at
256 px (lr raised to 1.0 for the short budget).

In [9]:
MACO_PAPER = dict(total_steps=4096, learning_rate=0.1, image_size=2048, model_input_size=224, noise=0.08,
                  crops_per_iteration=12, box_size=(0.2, 0.25), values_range=(0.0, 1.0))
MACO = dict(MACO_PAPER) if FULL_SCALE else dict(MACO_PAPER, total_steps=100, learning_rate=1.0, image_size=256, crops_per_iteration=4)
rng = np.random.RandomState(0)
template_imgs = [U.stimulus_path(n) for n in rng.choice(names, 64, replace=False)]
magnitude = U.natural_magnitude_template(template_imgs, MACO['image_size'])   # fixed |FFT| template; only the phase is optimized
t0 = time.time()
mres = U.maco(predict, magnitude, device='cpu', verbose=False, **MACO)
print(f'MACO: {MACO["total_steps"]} steps in {time.time() - t0:.0f} s; final full-canvas prediction {mres["history"][-1]:+.2f} (natural q99 = {q99:+.2f})')
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
axes[0].imshow(U.to_display(mres['image'])); axes[0].set_axis_off(); axes[0].set_title(f'MACO super-stimulus, site {UNIT}')
tr = mres['transparency'].mean(0); axes[1].imshow(tr / tr.max(), cmap='gray'); axes[1].set_axis_off(); axes[1].set_title('accumulated |gradient| (transparency mask)')   # where the objective actually pushed pixels
axes[2].plot(mres['history']); axes[2].axhline(q99, color='gray', ls=':', label='natural q99'); axes[2].axhline(q01, color='gray', ls=':')
axes[2].set(xlabel='step', ylabel='predicted response'); axes[2].legend(fontsize=7)
plt.tight_layout(); display(plt.gcf()); plt.close()
fetcher.cleanup()                                        # remove the ingraph hook from the backbone

MACO: 100 steps in 60 s; final full-canvas prediction +0.66 (natural q99 = +3.55)


<Figure size 900x288 with 3 Axes>

FeatureFetcher hooks all freed
